# 01 — Data Loading & Inspection
---
**Purpose:** Locate and safely load all 2022–2025 F1 lap data. Inspect schemas, standardize team names, preserve raw data, and perform initial dataset diagnostics.

**Output:** `outputs/combined_laps.parquet`

In [1]:
import pandas as pd
import numpy as np
import os, glob, warnings
warnings.filterwarnings("ignore")

DATA_DIR = os.path.join("..", "data_fastf1_v1", "laps")
OUTPUT_DIR = os.path.join("..", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEASONS = [2022, 2023, 2024, 2025]
print("Setup complete. Output directory ready.")

Setup complete. Output directory ready.


## 1. Locate Datasets & Inspect Schemas

In [2]:
all_schemas = {}
file_counts = {}
all_files = []

for season in SEASONS:
    season_dir = os.path.join(DATA_DIR, str(season))
    count = 0
    for csv_file in sorted(glob.glob(os.path.join(season_dir, "*.csv"))):
        fname = os.path.basename(csv_file)
        hdr = pd.read_csv(csv_file, nrows=0)
        all_schemas[f"{season}/{fname}"] = set(hdr.columns)
        all_files.append(csv_file)
        count += 1
    file_counts[season] = count
    print(f"  {season}: {count} files located.")

unique_sets = {}
for key, cols in all_schemas.items():
    frozen = frozenset(cols)
    unique_sets.setdefault(frozen, []).append(key)

print(f"\nTotal files: {len(all_files)}")
print(f"Distinct schemas found: {len(unique_sets)}")
if len(unique_sets) == 1:
    ref_cols = list(list(unique_sets.keys())[0])
    print(f"[OK] Schema is 100% consistent. All {len(all_files)} files share the exact same {len(ref_cols)}-column schema.")
else:
    print("[WARNING] Schema differences detected!")

  2022: 22 files located.


  2023: 22 files located.


  2024: 24 files located.


  2025: 24 files located.

Total files: 92
Distinct schemas found: 1
[OK] Schema is 100% consistent. All 92 files share the exact same 65-column schema.


## 2. Bahrain Demo Dataset Comparison

In [3]:
demo_path = os.path.join(DATA_DIR, "2024", "Bahrain_Grand_Prix.csv")
if os.path.exists(demo_path):
    demo_df = pd.read_csv(demo_path)
    print(f"Bahrain 2024 Demo Dataset:")
    print(f"  Rows: {len(demo_df):,}")
    print(f"  Cols: {len(demo_df.columns)}")
    
    # Pick a random other file to compare
    other_path = os.path.join(DATA_DIR, "2023", "Monaco_Grand_Prix.csv")
    other_df = pd.read_csv(other_path)
    
    print(f"\nComparison with Monaco 2023:")
    print(f"  Columns match? {set(demo_df.columns) == set(other_df.columns)}")
    
    demo_types = demo_df.dtypes.to_dict()
    other_types = other_df.dtypes.to_dict()
    type_diff = {k: (demo_types[k], other_types[k]) for k in demo_types if demo_types[k] != other_types[k]}
    
    if not type_diff:
        print("  Data types match 100% between the files.")
    else:
        print(f"  Type differences detected: {type_diff}")
else:
    print("Bahrain 2024 Demo file not found.")

Bahrain 2024 Demo Dataset:
  Rows: 1,129
  Cols: 65



Comparison with Monaco 2023:
  Columns match? True
  Type differences detected: {'IsPersonalBest': (dtype('bool'), dtype('O'))}


## 3. Load All Data (Safely)

In [4]:
all_dfs = []
for f in all_files:
    # Read without assuming types to avoid inference errors, then we'll cast
    all_dfs.append(pd.read_csv(f))

df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataset loaded: {len(df):,} laps x {len(df.columns)} columns")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Combined dataset loaded: 101,290 laps x 65 columns
Memory footprint: 78.4 MB


## 4. Duplicate Check

In [5]:
duplicates = df.duplicated().sum()
if duplicates == 0:
    print("[OK] Zero exact duplicate rows found across the entire dataset.")
else:
    print(f"[WARNING] Found {duplicates:,} duplicate rows!")

[OK] Zero exact duplicate rows found across the entire dataset.


## 5. Type Summaries & Casting

In [6]:
numeric_cols = [
    "LapNumber", "Stint", "TyreLife", "Position",
    "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
    "AirTemp", "Humidity", "Pressure", "TrackTemp", "WindSpeed",
    "LapTimeSeconds", "PitInTimeSeconds", "PitOutTimeSeconds",
    "Sector1TimeSeconds", "Sector2TimeSeconds", "Sector3TimeSeconds",
    "TimeSeconds", "LapStartTimeSeconds",
    "GapToLeaderSeconds", "IntervalToPositionAheadSeconds",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

bool_cols = [
    "IsPersonalBest", "FreshTyre", "Deleted", "FastF1Generated",
    "IsAccurate", "Rainfall", "HasGreen", "HasYellow",
    "HasSafetyCar", "HasRedFlag", "HasVSC", "HasVSCEnding",
]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

print("Type casting complete. Summary of Data Types:")
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"  {dtype}: {count} columns")

Type casting complete. Summary of Data Types:
  float64: 27 columns
  str: 20 columns
  bool: 12 columns
  int64: 5 columns
  object: 1 columns


## 6. Standardize Schema Inconsistencies

In [7]:
# 1. Drop dead column
dead_cols = [col for col in df.columns if df[col].isna().all()]
print(f"Fully null columns: {dead_cols}")
if "LapStartDate" in df.columns:
    df = df.drop(columns=["LapStartDate"])
    print("Dropped: LapStartDate")

# 2. Create RaceId
df["RaceId"] = df["Year"].astype(str) + "_" + df["GrandPrix"].str.replace(" ", "_")

# 3. Canonical Team Mapping
TEAM_MAPPING = {
    "Alfa Romeo": "Sauber", "Kick Sauber": "Sauber",
    "AlphaTauri": "VCARB", "RB": "VCARB", "Racing Bulls": "VCARB",
    "Alpine": "Alpine", "Aston Martin": "Aston Martin",
    "Ferrari": "Ferrari", "Haas F1 Team": "Haas",
    "McLaren": "McLaren", "Mercedes": "Mercedes",
    "Red Bull Racing": "Red Bull", "Williams": "Williams",
}
df["CanonicalTeam"] = df["Team"].map(TEAM_MAPPING)

print("\nStandardization complete. (Raw data preserved, mapping added).")

Fully null columns: ['LapStartDate']
Dropped: LapStartDate

Standardization complete. (Raw data preserved, mapping added).


## 7. Dataset Summary & Counts

In [8]:
print("=" * 60)
print("GLOBAL DATASET SUMMARY")
print("=" * 60)
print(f"  Total laps:       {len(df):,}")
print(f"  Unique races:     {df['RaceId'].nunique()}")
print(f"  Seasons:          {sorted(df['Year'].unique())}")
print(f"  Unique drivers:   {df['Driver'].nunique()}")
print(f"  Canonical teams:  {df['CanonicalTeam'].nunique()} (from {df['Team'].nunique()} raw names)")
print(f"  Unique circuits:  {df['GrandPrix'].nunique()}")
print(f"  Compounds:        {sorted(df['Compound'].dropna().unique())}")

print(f"\nLaps per season:")
for season in SEASONS:
    s = df[df["Year"] == season]
    print(f"  {season}: {len(s):,} laps, {s['RaceId'].nunique()} races")

GLOBAL DATASET SUMMARY
  Total laps:       101,290
  Unique races:     92
  Seasons:          [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Unique drivers:   31
  Canonical teams:  10 (from 13 raw names)
  Unique circuits:  25
  Compounds:        ['HARD', 'INTERMEDIATE', 'MEDIUM', 'SOFT', 'WET']

Laps per season:
  2022: 23,577 laps, 22 races


  2023: 24,420 laps, 22 races


  2024: 26,604 laps, 24 races
  2025: 26,689 laps, 24 races


## 8. Missing Value Summary

In [9]:
print(f"{'Column':<45} {'Nulls':>8}  {'%':>6}")
print("-" * 62)
for col in df.columns:
    null_n = df[col].isna().sum()
    if null_n > 0:
        print(f"  {col:<43} {null_n:>8,}  {100*null_n/len(df):>5.1f}%")

Column                                           Nulls       %
--------------------------------------------------------------
  LapTime                                        1,539    1.5%
  Stint                                            354    0.3%
  PitOutTime                                    97,823   96.6%
  PitInTime                                     97,821   96.6%
  Sector1Time                                    2,094    2.1%
  Sector2Time                                      169    0.2%
  Sector3Time                                      282    0.3%
  Sector1SessionTime                             2,317    2.3%
  Sector2SessionTime                               169    0.2%
  Sector3SessionTime                               282    0.3%
  SpeedI1                                          136    0.1%
  SpeedI2                                        1,020    1.0%
  SpeedFL                                        3,602    3.6%
  SpeedST                                          181 

## 9. Save Combined Standardized Dataset

In [10]:
output_path = os.path.join(OUTPUT_DIR, "data", "combined_laps.parquet")
df.to_parquet(output_path, index=False)
fsize = os.path.getsize(output_path) / 1e6
print(f"Saved dataset to: {output_path}")
print(f"File size: {fsize:.1f} MB")
print(f"Shape: {df.shape}")
print("\n[OK] Notebook 01 Data Loading & Inspection complete.")

Saved dataset to: ..\outputs\combined_laps.parquet
File size: 14.4 MB
Shape: (101290, 66)

[OK] Notebook 01 Data Loading & Inspection complete.
